<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">Lab 06: カスタム評価器 (Custom Grader) を作る</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Microsoft Foundry エージェント可観測性
  </p>
</div>

## 学習内容

[Lab 03a](lab-03a-tools.ipynb) では Contoso Travel エージェントに **Function Tools** を追加し、`search_flights` / `search_hotels` / `search_car_rentals` の 3 ツールを呼び出せるようにしました。エージェントは顧客のクエリに応じて適切なツールを選択します。

このラボでは、その **ツール選択の正確さ** を **カスタム評価器 (Custom Grader)** で定量評価します。

| フェーズ | 内容 |
|---------|------|
| **Phase 1** | 実ツール付きエージェントを動かし、実際のツール呼び出しデータを収集する |
| **Phase 2** | 収集したデータを 3 種類の Custom Grader で評価する |

Custom Grader の種類:

| グレーダー型 | 名前 | 評価内容 |
|------------|------|---------|
| `python` | `correct_tool_called` | 期待するツールを選択したか (0.0 / 1.0) |
| `python` | `required_params_present` | 必要なパラメーターが揃っているか (部分点あり) |
| `score_model` | `routing_quality` | LLM-as-judge によるルーティング判断の総合評価 (1–5) |

---

## 1. セットアップ

---


In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential, InteractiveBrowserCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, FunctionTool, Tool
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.responses.response_input_param import FunctionCallOutput

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

tenant_id = os.getenv("TENANT_ID")
endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential() if not tenant_id else AzureCliCredential(tenant_id=tenant_id)
#credential = InteractiveBrowserCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Microsoft Foundry に接続しました")
print(f"   モデル: {model_name}")

---

## Phase 1: 実ツール呼び出しデータの収集

Lab 03a と同じ手順で、実データを検索する **Function Tools 付きエージェント**を作成し、評価クエリに対して実行します。エージェントが返す `function_call` レスポンス (ツール名とパラメーター) を収集し、Phase 2 の評価データとして使います。

### 2. 旅行データの読み込みとツール関数の定義

---

In [ ]:
data_path = "../../data/contoso-travel"
flights_df = pd.read_csv(f"{data_path}/flights.csv")
hotels_df = pd.read_csv(f"{data_path}/hotels.csv")
cars_df = pd.read_csv(f"{data_path}/car_rentals.csv")

print(f"✈️  {len(flights_df)} 件のフライトを読み込みました")
print(f"🏨 {len(hotels_df)} 件のホテルを読み込みました")
print(f"🚗 {len(cars_df)} 件のレンタカーを読み込みました")


def search_flights(origin: str = None, destination: str = None, cabin_class: str = None, max_price: float = None) -> str:
    """Search for available flights based on criteria."""
    results = flights_df.copy()
    if origin:
        results = results[results["origin"].str.lower() == origin.lower()]
    if destination:
        results = results[results["destination"].str.lower() == destination.lower()]
    if cabin_class:
        results = results[results["cabin_class"].str.lower() == cabin_class.lower()]
    if max_price:
        results = results[results["price_usd"] <= max_price]
    if results.empty:
        return json.dumps({"message": "No flights found.", "results": []})
    return results.to_json(orient="records")


def search_hotels(city: str = None, min_stars: int = None, max_price: float = None, amenities: str = None) -> str:
    """Search for available hotels based on criteria."""
    results = hotels_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if min_stars:
        results = results[results["star_rating"] >= min_stars]
    if max_price:
        results = results[results["price_per_night_usd"] <= max_price]
    if amenities:
        results = results[results["amenities"].str.lower().str.contains(amenities.lower())]
    if results.empty:
        return json.dumps({"message": "No hotels found.", "results": []})
    return results.to_json(orient="records")


def search_car_rentals(city: str = None, car_type: str = None, max_price: float = None) -> str:
    """Search for available car rentals based on criteria."""
    results = cars_df.copy()
    if city:
        results = results[results["city"].str.lower() == city.lower()]
    if car_type:
        results = results[results["car_type"].str.lower() == car_type.lower()]
    if max_price:
        results = results[results["price_per_day_usd"] <= max_price]
    results = results[results["available"] == True]
    if results.empty:
        return json.dumps({"message": "No car rentals found.", "results": []})
    return results.to_json(orient="records")


print("\n✅ 3 つの検索関数を定義しました")

In [ ]:
flight_tool = FunctionTool(
    name="search_flights",
    description="Search for available flights. Use when customer asks about flights or airfare.",
    parameters={
        "type": "object",
        "properties": {
            "origin":       {"type": "string", "description": "Departure city"},
            "destination":  {"type": "string", "description": "Arrival city"},
            "cabin_class":  {"type": "string", "enum": ["Economy", "Business", "First"]},
            "max_price":    {"type": "number", "description": "Max price in USD"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

hotel_tool = FunctionTool(
    name="search_hotels",
    description="Search for available hotels. Use when customer asks about hotels or accommodation.",
    parameters={
        "type": "object",
        "properties": {
            "city":      {"type": "string"},
            "min_stars": {"type": "integer"},
            "max_price": {"type": "number"},
            "amenities": {"type": "string", "description": "e.g. Pool, Spa, WiFi"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

car_rental_tool = FunctionTool(
    name="search_car_rentals",
    description="Search for car rentals. Use when customer asks about rental cars.",
    parameters={
        "type": "object",
        "properties": {
            "city":     {"type": "string"},
            "car_type": {"type": "string", "enum": ["Economy", "SUV", "Luxury", "Minivan"]},
            "max_price": {"type": "number"},
        },
        "required": [],
        "additionalProperties": False,
    },
    strict=False,
)

tools: list[Tool] = [flight_tool, hotel_tool, car_rental_tool]

TRAVEL_AGENT_INSTRUCTIONS = """あなたは実際の旅行データにアクセスできる Contoso Travel エージェントです。
顧客の質問に答えるために適切なツールを使用してください。
必ず検索してから回答すること — 推測は禁止。"""

agent = project_client.agents.create_version(
    agent_name="contoso-travel-agent-eval",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=TRAVEL_AGENT_INSTRUCTIONS,
        tools=tools,
    ),
)

print(f"✅ ツール付きエージェントを作成しました: {agent.name} (v{agent.version})")
print(f"   登録済みツール: {[t.name for t in tools]}")

### 3. テストクエリの定義とツール呼び出しデータの収集

各クエリに対してエージェントを実行し、返ってきた `function_call` アイテム (ツール名とパラメーター) を収集します。これが Phase 2 の評価データになります。

---

In [ ]:
test_queries = [
    {
        "query": "8月にシアトルからパリへのフライトはありますか？",
        "expected_tool": "search_flights",
        "expected_params": {"origin": "Seattle", "destination": "Paris"},
    },
    {
        "query": "東京で温泉のあるホテルを見つけてください。",
        "expected_tool": "search_hotels",
        "expected_params": {"city": "Tokyo", "amenities": "Spa"},
    },
    {
        "query": "カンクンで格安のレンタカーが必要です。",
        "expected_tool": "search_car_rentals",
        "expected_params": {"city": "Cancun"},
    },
    {
        "query": "ニューヨーク行きのビジネスクラスのフライトを探してください。",
        "expected_tool": "search_flights",
        "expected_params": {"destination": "New York", "cabin_class": "Business"},
    },
    {
        "query": "パリの5つ星ホテルを予算300ドル以内で見つけてください。",
        "expected_tool": "search_hotels",
        "expected_params": {"city": "Paris", "min_stars": 5},
    },
    {
        "query": "ロサンゼルスでSUVのレンタカーを探しています。",
        "expected_tool": "search_car_rentals",
        "expected_params": {"city": "Los Angeles", "car_type": "SUV"},
    },
]

# エージェントを各クエリで実行し、function_call レスポンスを収集する
collected_items = []

for q in test_queries:
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input=q["query"],
    )

    for output_item in response.output:
        if output_item.type == "function_call":
            collected_items.append({
                "query":           q["query"],
                "actual_tool":     output_item.name,
                "actual_args":     output_item.arguments,   # JSON 文字列
                "expected_tool":   q["expected_tool"],
                "expected_params": json.dumps(q["expected_params"], ensure_ascii=False),
            })
            print(f"✅ {output_item.name}({output_item.arguments[:60]}...)")
            break  # 最初の function_call のみ収集

    openai_client.conversations.delete(conversation_id=conversation.id)

print(f"\n📋 {len(collected_items)} 件のツール呼び出しデータを収集しました")

### 3.1 JSONL ファイルとして保存・アップロード

収集したデータを `custom_evaluation_data.jsonl` に保存し、Foundry にアップロードします。アップロード済みファイルは評価実行で `file_id` として参照できます。

---

In [ ]:
jsonl_path = f"{data_path}/custom_evaluation_data.jsonl"

# JSONL 形式で保存: 1 行 = フラットな JSON オブジェクト ({"item": {...}} ラッパー不要)
with open(jsonl_path, "w", encoding="utf-8") as f:
    for ci in collected_items:
        f.write(json.dumps(ci, ensure_ascii=False) + "\n")

print(f"✅ JSONL を保存しました: {jsonl_path}")
print(f"   レコード数: {len(collected_items)}")

# Foundry Dataset として登録・アップロードする
dataset_name    = "contoso-travel-tool-calls"
dataset_version = "1"

dataset = project_client.datasets.upload_file(
    name=dataset_name,
    version=dataset_version,
    file_path=jsonl_path,
)
data_id = dataset.id

print(f"\n✅ データセットをアップロードしました")
print(f"   Dataset ID: {data_id}")
print(f"   Name: {dataset_name} (v{dataset_version})")

---

## Phase 2: カスタム評価器 (Custom Grader) で評価する

収集したツール呼び出しデータ (`actual_tool`, `actual_args`) を Custom Grader で評価します。グレーダーはすべて `item.*` フィールド (収集済みデータ) を参照するため、評価実行中に追加のモデル呼び出しを必要としません。

### 4. カスタム評価器の定義

#### 4.1 `python` グレーダー — `correct_tool_called` と `required_params_present`

`grade(sample, item)` 関数が Foundry のサンドボックスで実行されます。`item` には収集したツール呼び出しデータが入っています。

---

In [ ]:
# Python grader 1: 正しいツールが選ばれたか
correct_tool_grader_source = r"""
def grade(sample, item) -> float:
    return 1.0 if item.get("actual_tool") == item.get("expected_tool") else 0.0
"""

# Python grader 2: 必要なパラメーターが揃っているか (部分点あり)
required_params_grader_source = r"""
import json

def grade(sample, item) -> float:
    try:
        actual   = json.loads(item.get("actual_args",     "{}"))
        expected = json.loads(item.get("expected_params", "{}"))
        if not expected:
            return 1.0
        matched = sum(1 for k in expected if k in actual)
        return matched / len(expected)
    except (json.JSONDecodeError, AttributeError):
        return 0.0
"""

print(correct_tool_grader_source)
print(required_params_grader_source)

#### 4.2 `score_model` グレーダー — `routing_quality`

LLM-as-judge です。顧客のクエリと実際に呼び出されたツール・パラメーターを見て、ルーティング判断の適切さを 1〜5 で採点します。`correct_tool_called` が binary (0/1) なのに対し、こちらは**微妙な判断ミス**を段階的に評価できます。

---

In [ ]:
custom_data_source_config = DataSourceConfigCustom(
    type="custom",
    item_schema={
        "type": "object",
        "properties": {
            "query":           {"type": "string"},
            "actual_tool":     {"type": "string"},
            "actual_args":     {"type": "string"},
            "expected_tool":   {"type": "string"},
            "expected_params": {"type": "string"},
        },
        "required": ["query", "actual_tool", "actual_args", "expected_tool", "expected_params"],
    },
    include_sample_schema=False,  # grader は item.* のみ使用する
)

custom_testing_criteria = [
    # (1) Python grader: 正しいツールが選ばれたか
    {
        "type": "python",
        "name": "correct_tool_called",
        "source": correct_tool_grader_source,
        "pass_threshold": 1.0,
    },
    # (2) Python grader: 必要なパラメーターが揃っているか
    {
        "type": "python",
        "name": "required_params_present",
        "source": required_params_grader_source,
        "pass_threshold": 1.0,
    },
    # (3) Score model grader: LLM-as-judge によるルーティング総合評価 (1-5)
    {
        "type": "score_model",
        "name": "routing_quality",
        "model": model_name,
        "input": [
            {
                "role": "system",
                "content": (
                    "あなたは旅行 AI システムの品質審査員です。"
                    "顧客の質問に対してエージェントが選んだツールとパラメーターの"
                    "適切さを 1〜5 の整数で評価し、数値のみを返してください。\n"
                    "5 = 完璧、4 = ほぼ適切、3 = 普通、2 = やや不適切、1 = 完全に間違い"
                ),
            },
            {
                "role": "user",
                "content": (
                    "顧客の質問: {{item.query}}\n"
                    "呼び出されたツール: {{item.actual_tool}}\n"
                    "パラメーター: {{item.actual_args}}"
                ),
            },
        ],
        "range": [1, 5],
        "pass_threshold": 4.0,
    },
]

custom_eval = openai_client.evals.create(
    name="Contoso Travel - Tool Call Custom Graders",
    data_source_config=custom_data_source_config,
    testing_criteria=custom_testing_criteria,
)

print(f"✅ カスタム評価を作成しました (ID: {custom_eval.id})")
for c in custom_eval.testing_criteria:
    print(f"   • {c.name} ({c.type})")

### 5. カスタム評価の実行

アップロードしたデータセットを `file_id` で参照して評価を実行します。グレーダーは `item.*` フィールドのみを参照するため、`type: "jsonl"` の **Dataset evaluation** モードを使用します。エージェントへの追加呼び出しは発生しません。

---

In [ ]:
custom_data_source = {
    "type": "jsonl",
    "source": {
        "type": "file_id",
        "id": data_id,
    },
}

custom_run = openai_client.evals.runs.create(
    eval_id=custom_eval.id,
    name="Tool Call Grader Run - Contoso Travel",
    data_source=custom_data_source,
)

print(f"✅ 評価実行を開始しました (ID: {custom_run.id})")
print(f"   ステータス: {custom_run.status}")

while custom_run.status not in ["completed", "failed"]:
    custom_run = openai_client.evals.runs.retrieve(
        run_id=custom_run.id, eval_id=custom_eval.id
    )
    print(f"   ⏳ Status: {custom_run.status}")
    time.sleep(5)

print(f"\n{'✅' if custom_run.status == 'completed' else '❌'} 評価が {custom_run.status} になりました！")

### 6. 結果の解釈

| 評価器 | スコアの意味 | レッドフラグ |
|--------|-------------|-------------|
| `correct_tool_called`     | 1.0 = 正解 / 0.0 = 不正解               | 0.0 → 間違ったツールを選んでいる |
| `required_params_present` | 0.0〜1.0 (パラメーター一致率)           | 0.5 以下 → 必要なパラメーターが欠けている |
| `routing_quality`         | 1〜5 (4 以上で合格)                     | 3 以下 → ルーティング判断が不適切 |

---

In [ ]:
if custom_run.status == "completed":
    print(f"🛠️ カスタム評価結果")
    print(f"   結果カウント: {custom_run.result_counts}")

    output_items = list(
        openai_client.evals.runs.output_items.list(
            run_id=custom_run.id, eval_id=custom_eval.id
        )
    )

    print(f"\n{'='*70}")
    for item in output_items:
        q            = item.datasource_item.get("query", "N/A")
        actual_tool  = item.datasource_item.get("actual_tool", "N/A")
        print(f"\nQuery:  {q[:60]}")
        print(f"Tool:   {actual_tool}")
        if hasattr(item, "results") and item.results:
            for result in item.results:
                name   = getattr(result, "name", "N/A")
                score  = getattr(result, "score", "N/A")
                passed = getattr(result, "passed", "N/A")
                print(f"  {name:28s}: score={score}, passed={passed}")
    print(f"{'='*70}")
else:
    print("❌ 評価が失敗しました。Foundry ポータルで詳細を確認してください。")

### 7. Foundry ポータルで結果を確認する

1. [Microsoft Foundry ポータル](https://ai.azure.com) にアクセス
2. プロジェクトを開く
3. 左ナビゲーションの **Evaluations** タブをクリック
4. **Contoso Travel - Tool Call Custom Graders** の実行をクリック
5. 3 つのカスタム評価器ごとに、クエリ単位のスコアと判定理由 (LLM-as-judge の場合) を確認できます

---

## 8. クリーンアップ

---


In [ ]:
openai_client.evals.delete(eval_id=custom_eval.id)
print("✅ カスタム評価を削除しました")

project_client.datasets.delete_version(name=dataset_name, version=dataset_version)
print("✅ データセットを削除しました")

project_client.agents.delete(agent_name=agent.name)
print("✅ エージェントを削除しました")

openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントを閉じました")